# Convert COGs To Zarr

COG is excellent for individual rasters. Zarr is often better when many rasters form a logical multidimensional collection, for example one COG per depth or one COG per time step. This notebook shows both a single-layer conversion and a depth-stack conversion.


In [1]:
from pathlib import Path


def find_repo_root(start: Path = Path.cwd()) -> Path:
    """Find the repository root when the kernel starts in a subfolder."""
    start = start.resolve()
    for path in (start, *start.parents):
        if (path / ".git").exists() or (path / "downloaded_data").exists():
            return path
    return start


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "downloaded_data"
DATA_DIR.mkdir(exist_ok=True)

import re

import matplotlib.pyplot as plt
import numpy as np
import rioxarray
import xarray as xr

BASAL_MELT_COG = DATA_DIR / "basal_melt_map_racmo_firn_air_corrected_cog.tif"
BASAL_MELT_ZARR = DATA_DIR / "basal_melt_map_racmo_firn_air_corrected_cog.zarr"
ICE_TEMP_ZARR = DATA_DIR / "Tice_EPSG3031_depth_stack.zarr"

## Single COG To Single-Layer Zarr


In [3]:
da = rioxarray.open_rasterio(BASAL_MELT_COG, masked=True, chunks={"x": 512, "y": 512, "band": 1})
basal_melt = da.isel(band=0, drop=True).to_dataset(name="basal_melt")
basal_melt["basal_melt"].attrs.update({"source_cog": BASAL_MELT_COG.name})
basal_melt["basal_melt"].attrs.pop("grid_mapping", None)
basal_melt

<xarray.Dataset> Size: 158MB
Dimensions:      (x: 6120, y: 6435)
Coordinates:
  * x            (x) float64 49kB -3.06e+06 -3.059e+06 ... 3.058e+06 3.059e+06
  * y            (y) float64 51kB 3.217e+06 3.216e+06 ... -3.216e+06 -3.217e+06
    spatial_ref  int64 8B 0
Data variables:
    basal_melt   (y, x) float32 158MB dask.array<chunksize=(512, 512), meta=np.ndarray>

In [4]:
basal_melt.to_zarr(BASAL_MELT_ZARR, mode="w", consolidated=True)

/home/krasen/micromamba/envs/pangeo/lib/python3.13/site-packages/zarr/api/asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


## Stack Depth COGs Into One Zarr

The depth value is parsed from filenames like `Tice_EPSG3031_depth_500m.tif`. This keeps the stack robust if the selected depths change.


In [7]:
DEPTH_RE = re.compile(r"depth_([0-9]+(?:\.[0-9]+)?)m", re.IGNORECASE)


def parse_depth_m(path):
    match = DEPTH_RE.search(path.name)
    if not match:
        raise ValueError(f"Cannot parse depth from {path.name}")
    return float(match.group(1))


depth_cogs = sorted(DATA_DIR.glob("Tice_EPSG3031_depth_*m.tif"), key=parse_depth_m)
depth_cogs

[PosixPath('/home/krasen/hackathon-repo/downloaded_data/Tice_EPSG3031_depth_0m.tif'),
 PosixPath('/home/krasen/hackathon-repo/downloaded_data/Tice_EPSG3031_depth_500m.tif'),
 PosixPath('/home/krasen/hackathon-repo/downloaded_data/Tice_EPSG3031_depth_1000m.tif'),
 PosixPath('/home/krasen/hackathon-repo/downloaded_data/Tice_EPSG3031_depth_1500m.tif'),
 PosixPath('/home/krasen/hackathon-repo/downloaded_data/Tice_EPSG3031_depth_2000m.tif'),
 PosixPath('/home/krasen/hackathon-repo/downloaded_data/Tice_EPSG3031_depth_3000m.tif')]

In [8]:
arrays = []
for path in depth_cogs:
    depth_m = parse_depth_m(path)
    layer = rioxarray.open_rasterio(path, masked=True, chunks={"x": 512, "y": 512})
    layer = layer.isel(band=0, drop=True).assign_coords(depth=depth_m).expand_dims("depth")
    layer.name = "Tice"
    arrays.append(layer)

stack = xr.concat(arrays, dim="depth").sortby("depth").to_dataset(name="Tice")
stack["Tice"].attrs.update({"units": "K", "source_layout": "one COG per depth"})
stack

<xarray.Dataset> Size: 689MB
Dimensions:      (x: 5672, y: 5062, depth: 6)
Coordinates:
  * x            (x) float64 45kB -2.836e+06 -2.834e+06 ... 2.834e+06 2.836e+06
  * y            (y) float64 40kB 2.53e+06 2.53e+06 ... -2.53e+06 -2.53e+06
    spatial_ref  int64 8B 0
  * depth        (depth) float64 48B 0.0 500.0 1e+03 1.5e+03 2e+03 3e+03
Data variables:
    Tice         (depth, y, x) float32 689MB dask.array<chunksize=(1, 512, 512), meta=np.ndarray>

In [12]:
# rechunk the data
stack = stack.chunk({'depth': 2, 'y':5063, 'x':5673})
stack

<xarray.Dataset> Size: 689MB
Dimensions:      (x: 5672, y: 5062, depth: 6)
Coordinates:
  * x            (x) float64 45kB -2.836e+06 -2.834e+06 ... 2.834e+06 2.836e+06
  * y            (y) float64 40kB 2.53e+06 2.53e+06 ... -2.53e+06 -2.53e+06
    spatial_ref  int64 8B 0
  * depth        (depth) float64 48B 0.0 500.0 1e+03 1.5e+03 2e+03 3e+03
Data variables:
    Tice         (depth, y, x) float32 689MB dask.array<chunksize=(2, 5062, 5672), meta=np.ndarray>

In [13]:
stack.to_zarr(ICE_TEMP_ZARR, mode="w", consolidated=True)

/home/krasen/micromamba/envs/pangeo/lib/python3.13/site-packages/zarr/api/asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
